# ***Removing Corrupt Data***

### 1) counting rows

In [1]:
SELECT COUNT(*) FROM Practice_Schema.historical_live

StatementMeta(, e6cb88d3-c192-4031-90ac-bfcd777436f4, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

### 2) Checking Schema

In [5]:
%%pyspark
df = spark.table("Practice_Schema.historical_live")
df.printSchema()

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 10, Finished, Available, Finished, False)

root
 |-- timestamp: string (nullable = true)
 |-- ticker: string (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)



### 3.0) Null Value Check

In [7]:
%%pyspark
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 13, Finished, Available, Finished, False)

In [31]:
%%pyspark
df2 = df.filter(  (col("open").isNull()) & (col("volume").isNotNull() ) )
df2.select(count("*")).show()

StatementMeta(, d05f920c-3490-486b-a904-2893a7411bf1, 34, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|     301|
+--------+



In [4]:
%%pyspark
df.filter(   (col("close").isNull())  & (col("volume").isNull()))\
  .select(count("*")).show()

null_records_df = df.filter(col("close").isNull())
null_records_df.show()
null_records_df.select(count("*")).show()

  # 19056 rows were all ohlcv values are nulls
  # 301 rows where all ohlcv values except volume are all null

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 8, Finished, Available, Finished, False)

NameError: name 'df' is not defined

### 3.1) **Storing 19357 records in null_records**

In [50]:
%%pyspark
null_records_df.write.format("delta") \
               .mode("overwrite")\
               .saveAsTable("Silver.Corrupt_Schema.null_records")

StatementMeta(, d05f920c-3490-486b-a904-2893a7411bf1, 109, Finished, Available, Finished, False)

In [102]:
%%pyspark
null_records_df.write.format("delta") \
               .mode("overwrite")\
               .saveAsTable("Silver.Corrupt_Schema.total_rejected_records")

StatementMeta(, d05f920c-3490-486b-a904-2893a7411bf1, 229, Finished, Available, Finished, False)

### 3.2) **Removing Null records from dataframe**

In [8]:
%%pyspark
updated_df_1 = df.filter(col("close").isNotNull())
updated_df_1.show()
updated_df_1.select(count("*")).show()
#80639 records left after removing 19357 null records

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 14, Finished, Available, Finished, False)

+----------+-----------+------------------+------------------+------------------+------------------+-----------+
| timestamp|     ticker|              open|              high|               low|             close|     volume|
+----------+-----------+------------------+------------------+------------------+------------------+-----------+
|2021-08-07|    ETH-USD|  2891.70751953125| 3170.229736328125|  2868.53564453125|  3157.23876953125|33081467129|
|2021-08-07|    BTC-USD|      42832.796875|      44689.859375|    42618.56640625|    44555.80078125|40030862141|
|2021-08-08|    ETH-USD| 3161.232666015625|  3184.60400390625| 2951.747314453125| 3013.732666015625|28433638008|
|2021-08-08|    BTC-USD|        44574.4375|     45282.3515625|    43331.91015625|     43798.1171875|36302664750|
|2021-08-09|      ^GSPC|  4437.77001953125|  4439.39013671875|    4424.740234375|  4432.35009765625| 3449280000|
|2021-08-09|        BAC|  35.4347059759668| 36.23983900082299|  35.2135155474338| 35.98325729370

### 4.0) **Checking for negative records and 0 value records**

In [2]:
%%pyspark
updated_df_1.filter(  (col("volume")==0) )\
.select(count("*")).show()

updated_df_1.filter(  (col("open")<=0)   
                    | (col("close")<=0)
                    | (col("low")<=0) 
                    | (col("high")<=0))\
.select(count("*")).show()

updated_df_1.filter(  (col("open")<=0)   
                    | (col("close")<=0)
                    | (col("low")<=0) 
                    | (col("high")<=0)
                    | (col("volume")<=0))\
.select(count("*")).show()

#3400 records where volume==0
#3150 records where any ohlcv is less than 0
# thus total 6268 records which have ohlcv<0 and volume==0 (some records are common so 3400+3150=3550 is not applicable)
# volume doesnt have any negative record

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 5, Finished, Available, Finished, False)

NameError: name 'updated_df_1' is not defined

### 4.1) **Storing these negative/zero values in negative_zero_records**

In [100]:
%%pyspark
negative_zero_df = updated_df_1.filter(  (col("open")<=0)   
                    | (col("close")<=0)
                    | (col("low")<=0) 
                    | (col("high")<=0)
                    | (col("volume")<=0))
                    
negative_zero_df.select(count("*")).show()

StatementMeta(, d05f920c-3490-486b-a904-2893a7411bf1, 226, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|    6268|
+--------+



In [101]:
%%pyspark
negative_zero_df.write.format("delta") \
               .mode("overwrite")\
               .saveAsTable("Silver.Corrupt_Schema.nega_zero_records")

StatementMeta(, d05f920c-3490-486b-a904-2893a7411bf1, 228, Finished, Available, Finished, False)

In [104]:
%%pyspark
negative_zero_df.write.format("delta") \
               .mode("append")\
               .saveAsTable("Silver.Corrupt_Schema.total_rejected_records")


StatementMeta(, d05f920c-3490-486b-a904-2893a7411bf1, 232, Finished, Available, Finished, False)

In [13]:
SELECT COUNT(*) FROM Silver.Corrupt_Schema.total_rejected_records

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

### 4.2) **Removing nega/0 value records**

In [16]:
%%pyspark

updated_df_2 = updated_df_1.filter(  (col("open")>0)   
                    & (col("close")>0)
                    & (col("low")>0) 
                    & (col("high")>0)
                    & (col("volume")>0))
updated_df_2.show()
updated_df_2.select(count("*")).show()
# 74371 records left after removing 6268 from 80639

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 43, Finished, Available, Finished, False)

+----------+-----------+------------------+------------------+------------------+------------------+-----------+
| timestamp|     ticker|              open|              high|               low|             close|     volume|
+----------+-----------+------------------+------------------+------------------+------------------+-----------+
|2021-08-07|    ETH-USD|  2891.70751953125| 3170.229736328125|  2868.53564453125|  3157.23876953125|33081467129|
|2021-08-07|    BTC-USD|      42832.796875|      44689.859375|    42618.56640625|    44555.80078125|40030862141|
|2021-08-08|    ETH-USD| 3161.232666015625|  3184.60400390625| 2951.747314453125| 3013.732666015625|28433638008|
|2021-08-08|    BTC-USD|        44574.4375|     45282.3515625|    43331.91015625|     43798.1171875|36302664750|
|2021-08-09|      ^GSPC|  4437.77001953125|  4439.39013671875|    4424.740234375|  4432.35009765625| 3449280000|
|2021-08-09|        BAC|  35.4347059759668| 36.23983900082299|  35.2135155474338| 35.98325729370

### 5.0) **No of Duplicates**

In [28]:
%%pyspark
total_rows = updated_df_2.count()
unique_rows = updated_df_2.dropDuplicates().count()

print(total_rows)
print(unique_rows)

print(total_rows-unique_rows)

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 97, Finished, Available, Finished, False)

74371
55261
19110


### 5.1) **Removing Duplicates**

In [48]:
%%pyspark
updated_df_3 = updated_df_2.dropDuplicates()





StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 119, Finished, Available, Finished, False)

### 5.2) **Getting Duplicates**

In [35]:
%%pyspark
duplicate_df = updated_df_2.exceptAll(updated_df_3)
duplicate_df.select(count("*").alias("Duplicate_records")).show()


StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 106, Finished, Available, Finished, False)

+-----------------+
|Duplicate_records|
+-----------------+
|            19110|
+-----------------+



### 5.3) **Storing Duplicate Records**

In [37]:
%%pyspark
duplicate_df.write.format("delta")\
                  .mode("overwrite")\
                  .saveAsTable("Silver.Corrupt_Schema.duplicate_records")
                  

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 108, Finished, Available, Finished, False)

In [38]:
SELECT COUNT(*) FROM Silver.Corrupt_Schema.duplicate_records

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 109, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [39]:
%%pyspark
duplicate_df.write.format("delta")\
                  .mode("append")\
                  .saveAsTable("Silver.Corrupt_Schema.total_rejected_records")

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 110, Finished, Available, Finished, False)

In [41]:
SELECT COUNT(*) AS Total_rejected_Records FROM Silver.Corrupt_Schema.total_rejected_records
-- 19357 + 6268 + 19110 = 44735

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 112, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

# **Standardizing dataTypes**

### 6.0) **Checking Schema**

In [49]:
%%pyspark
updated_df_3.printSchema()

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 120, Finished, Available, Finished, False)

root
 |-- timestamp: string (nullable = true)
 |-- ticker: string (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)



### 6.1) **Changing string to timestamp**

In [56]:
%%pyspark
updated_df_4 = updated_df_3.withColumn(
    "timestamp",
    col("timestamp").cast("timestamp")
)
updated_df_4.printSchema()
updated_df_4.show()
# changed datatype of timestamp from string to timestamp

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 161, Finished, Available, Finished, False)

root
 |-- timestamp: timestamp (nullable = true)
 |-- ticker: string (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)

+-------------------+-----------+------------------+------------------+------------------+------------------+-----------+
|          timestamp|     ticker|              open|              high|               low|             close|     volume|
+-------------------+-----------+------------------+------------------+------------------+------------------+-----------+
|2021-08-18 00:00:00|       INTC|48.447813907377075|48.843870473339166|47.996495922442186|  48.0701789855957|   15061700|
|2021-08-18 00:00:00|    ETH-USD| 3011.963623046875|  3124.97607421875|   2959.0283203125|     3020.08984375|21539248425|
|2021-08-27 00:00:00|    BTC-USD|     46894.5546875|    49112.78515625|       46394.28125|    49058.66796875|34511076995|


# ***Rounding Values***

### 7.0) **Rounding Values**

In [61]:
%%pyspark
updated_df_4.createOrReplaceTempView("updated_df_4_table")

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 168, Finished, Available, Finished, False)

In [66]:
%%pyspark
updated_df_5 = spark.sql("""
    SELECT timestamp, ticker,
     round(open,2) AS open,
      round(high,2) AS high, 
      round(low,2) AS low, 
      round(close,2) AS close,
       volume
FROM updated_df_4_table
""")

updated_df_5.show()
updated_df_5.select(count("*")).show()

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 226, Finished, Available, Finished, False)

+-------------------+-----------+--------+--------+--------+--------+-----------+
|          timestamp|     ticker|    open|    high|     low|   close|     volume|
+-------------------+-----------+--------+--------+--------+--------+-----------+
|2021-08-18 00:00:00|       INTC|   48.45|   48.84|    48.0|   48.07|   15061700|
|2021-08-18 00:00:00|    ETH-USD| 3011.96| 3124.98| 2959.03| 3020.09|21539248425|
|2021-08-27 00:00:00|    BTC-USD|46894.55|49112.79|46394.28|49058.67|34511076995|
|2021-09-09 00:00:00|HDFCBANK.NS|  736.47|  739.02|  730.39|  733.95|    8250948|
|2021-09-16 00:00:00|     ^BSESN|58881.04|59204.29| 58700.5|59141.16|      24700|
|2021-09-17 00:00:00|        CRM|  254.86|  257.25|  254.28|   256.1|    7049400|
|2021-09-24 00:00:00|       NFLX|   59.25|    59.3|   58.36|   59.24|   21262000|
|2021-10-01 00:00:00|       COST|   426.2|  427.48|   417.7|  424.87|    1860700|
|2021-10-04 00:00:00|       ADBE|  574.59|   576.8|  552.14|  558.49|    3977000|
|2021-10-08 00:0

# *Creating Year and Month*

In [78]:
%%pyspark
updated_df_6 = updated_df_5.withColumn("year", year("timestamp"))\
                           .withColumn("month", month("timestamp"))\
                           .withColumn("day_of_week", dayofweek("timestamp"))\
                           .withColumn("day_of_month", dayofmonth("timestamp"))
                           
updated_df_6.show()

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 255, Finished, Available, Finished, False)

+-------------------+-----------+--------+--------+--------+--------+-----------+----+-----+-----------+------------+
|          timestamp|     ticker|    open|    high|     low|   close|     volume|year|month|day_of_week|day_of_month|
+-------------------+-----------+--------+--------+--------+--------+-----------+----+-----+-----------+------------+
|2021-08-18 00:00:00|       INTC|   48.45|   48.84|    48.0|   48.07|   15061700|2021|    8|          4|          18|
|2021-08-18 00:00:00|    ETH-USD| 3011.96| 3124.98| 2959.03| 3020.09|21539248425|2021|    8|          4|          18|
|2021-08-27 00:00:00|    BTC-USD|46894.55|49112.79|46394.28|49058.67|34511076995|2021|    8|          6|          27|
|2021-09-09 00:00:00|HDFCBANK.NS|  736.47|  739.02|  730.39|  733.95|    8250948|2021|    9|          5|           9|
|2021-09-16 00:00:00|     ^BSESN|58881.04|59204.29| 58700.5|59141.16|      24700|2021|    9|          5|          16|
|2021-09-17 00:00:00|        CRM|  254.86|  257.25|  254

# Partition by year and month

In [83]:
%%pyspark
updated_df_6.write.format("delta") \
                   .partitionBy("year", "month") \
                   .mode("overwrite") \
                   .saveAsTable("silver.Cleanse_data.cleaned_stock_data")
# we have created 1 delta table and partioning can be seen through viewing files of the delta table
#By partitioning your Silver Delta table by year and month, you've implemented partition pruning, which reduces the amount of data Spark scans.

StatementMeta(, 98152e64-6f50-48fc-9e0a-aa125cdfd37b, 263, Finished, Available, Finished, False)